### Imports

In [70]:
# import the necessary packages

import os
from dotenv import load_dotenv

# langchain imports
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq


In [53]:
# initialize the object
load_dotenv()

True

In [55]:
# load the secret API keys

groq_key = os.getenv("GROQ_API_KEY")
jina_api_key = os.getenv("JINA_API_KEY")

print("Environment variables loaded!")

Environment variables loaded!


### Loading the data

In [11]:
data_file_path = os.path.join("data","hr_policy.txt")

print(data_file_path)

data\hr_policy.txt


# Data Ingestion

### Data Loading

In [14]:
# make an object of text loader class

loader = TextLoader(data_file_path, encoding = "utf-8")

documents = loader.load()

print(documents)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDuring probati

In [30]:
# print the metadata/page_content

print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [33]:
# display the total number of characters

print(f"The total number of characters is {len(documents[0].page_content)}.")

The total number of characters is 2597.


### Data Splitting

In [36]:
# make a variable
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents)

In [37]:
# display the chunks
print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [38]:
# display the number of chunks
print(f"chunks: {len(chunks)}")

chunks: 9


In [ ]:
# display the chunk data
print(f"number of characters in the chunk: {len(chunks[0].page_content)}")
print(chunks[0].page_content)
print(chunks[0].metadata)


number of characters in the chunk: 72
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)
{'source': 'data\\hr_policy.txt'}


In [43]:
# display the chunk data for the last chunk
print(f"number of characters in the chunk: {len(chunks[-1].page_content)}")
print(chunks[-1].page_content)
print(chunks[-1].metadata)

number of characters in the chunk: 303
8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.
{'source': 'data\\hr_policy.txt'}


### Create Vector Embeddings using JINA AI

In [59]:
# using JINA open-source free embeddings model

embeddings_model = JinaEmbeddings(model_name ="jina-embeddings-v2-base-en")

print(f"The model: {embeddings_model.model_name} sucessfully created vector embeddings!")

The model: jina-embeddings-v2-base-en sucessfully created vector embeddings!


### Create Vector Database using FAISS

In [67]:
# instantiate vector db

vector_store = FAISS.from_documents(chunks,embeddings_model)

print("chunks are stored ",vector_store.index.ntotal)

chunks are stored  9


### Store the data in VectorDB

In [69]:
test_query = "How many sick leaves employees get"

## similarity search

top_matches = vector_store.similarity_search(test_query,k=2)
print(f"Quesry: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"---Match {i}---")
    print(match.page_content)
    print()


Quesry: How many sick leaves employees get

---Match 1---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

---Match 2---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



## Data Retrieval 

In [ ]:
# LLM implementation

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature = 0 # controls the creativity of the model, 0 is deterministic
)

In [72]:
# display the model name
llm.model_name

'openai/gpt-oss-20b'

In [74]:
# test responses

test_response = llm.invoke("Hey, is learning RAG hard, answer in one line.")

In [77]:
# display the LLM answer

test_response.content

'Learning RAG is manageable with a solid NLP and coding foundation, but it can be challenging if you’re new to retrieval or transformer models.'

### Add a RETRIEVER TOOl

In [90]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})


def search_hr_policy(question:str) -> str:
    """Search the HR policies related to leave, WFH, probation, notice period, holidays, code of conduct"""
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

### Components of AI Agent

1. LLM: the brain
2. Tool: extra capabilities to make the model more capable and powerful
3. Memory: the persistence component, helps to recall and remember past conversations

In [78]:
# create an agent
from langchain.agents import create_agent

In [91]:
hr_assistant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt="""

    You are a friendly HR assistant.
    Always use the search_hr_policy tool to look up the facts before answering.
    If the answer isn't in the search results, say you don't know, no guessing allowed.

    """
)

print('HR assistant is ready.')

HR assistant is ready.


In [92]:
def ask_hr_assistant(question:str)-> str:
    """Send a question to the RAG and print a nicely formatted answer"""

    print("="*60)
    print("QUESTION: ", question)
    print("="*60)

    response = hr_assistant.invoke({"messages":[{"role":"user","content":question}]})
    answer = response["messages"][-1].content

    print("ANSWER: ", answer)
    print("="*60)
    print()

    return answer


In [100]:
response = hr_assistant.invoke({"messages":[{"role":"user","content":"tell me about leave policies how to apply for leave"}]})


In [94]:
response

{'messages': [HumanMessage(content='tell me about leave policies how to apply for leave', additional_kwargs={}, response_metadata={}, id='ffa1d86b-c0bc-4d4e-9046-07d561065652'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use search_hr_policy tool.', 'tool_calls': [{'id': 'fc_f459ae37-a005-43f4-8684-941c457ee108', 'function': {'arguments': '{"question":"leave policies how to apply for leave"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 193, 'total_tokens': 233, 'completion_time': 0.040824209, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.00939458, 'prompt_tokens_details': None, 'queue_time': 0.318800486, 'total_time': 0.050218789}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_7d448090ba', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fa885-c45c-7ea1-bf71

In [101]:
# display the messages
print(f"AI-Message: {response["messages"][-1].content}")

AI-Message: **Leave Policies (as per the HR handbook)**  

| Type of Leave | Entitlement | Carry‑Forward | Application Notes |
|---------------|-------------|---------------|-------------------|
| **Annual (Paid) Leave** | 20 days per calendar year (full‑time employees) | Up to **5 days** can be carried forward to the next year | Submit a request **at least 5 working days** in advance via the HR portal. |
| **Sick Leave** | 10 paid days per year | Not carried forward | For sick leave longer than **2 consecutive days**, a **medical certificate** must be uploaded with the request. |
| **Public Holidays** | 12 per year (as per the official calendar) | – | Working on a public holiday entitles you to compensatory leave. |

---

### How to Apply for Leave

1. **Log in to the HR Portal**  
   - Use your company credentials to access the portal.

2. **Navigate to the Leave Request Section**  
   - Usually found under “Time & Attendance” or “Leave Management”.

3. **Create a New Leave Request**